# IAT 461/882 — Assignment - Unsupervised Learning - Vancouver Business Licences Explorer

#### Submitted by:
Atif M. Mahmud  
atifm@sfu.ca

In [7]:
# Import the libraries
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as plt

## Part A - Business-level clustering

### Part A1 - Data acquisition and cleaning

#### Load and observe

In [8]:
gdf = gpd.read_file("data/business-licences.geojson")

print("The entire dataframe")
display(gdf)

print(f"\nThe shape of the dataframe is {gdf.shape}")
print(f"The columns of the dataframe are {gdf.columns.tolist()}")

print("\nDescriptive data")
display(gdf.describe(include="all"))

## NOTE: isna() and isnull() do the same thing, so I am using only one of them. I checked - no empty strings or "None" - or at least, they are handled by isna(). See commented code below
## To keep it simple, I will only use the isna()

# Null/Empty string/NaN/None: Fill in dataframe
# for column in gdf.columns:
#    df_clean_explore.at[column, "empty_string"] =  (gdf[column] == "").sum()
#    df_clean_explore.at[column, "nan_vals"] = gdf[column].isna().sum()
#    df_clean_explore.at[column, "none_vals"] = (gdf[column] == "None").sum()

print("\nGiven below: Column, NaN vals, Percentage of dataset")
rows = len(gdf)
for column in gdf.columns:
    nan_vals = gdf[column].isna().sum()
    percentage = (nan_vals/rows) * 100
    print(f"{column} | {nan_vals} | {percentage}%.")

The entire dataframe


,folderyear,licencersn,licencenumber,licencerevisionnumber,businessname,businesstradename,status,issueddate,expireddate,businesstype,...,province,country,postalcode,localarea,numberofemployees,feepaid,extractdate,geom,geo_point_2d,geometry
0,24,4518841,24-138560,10,Nobl Collective Ltd,NaN,Issued,2023-12-28 21:34:28+00:00,2024-12-31,Consulting and Management Services,...,BC,CA,NaN,Kitsilano,1.0,NaN,2026-07-01 02:32:17-07:00,None,NaN,None
1,24,4518843,24-138562,10,645064 BC Ltd,Trade Exchange Canada,Issued,2023-11-20 21:40:18+00:00,2024-12-31,Business Support Services,...,BC,CA,NaN,Kitsilano,1.0,NaN,2026-07-01 02:32:17-07:00,None,NaN,None
2,24,4518844,24-138563,10,(Louise Turgeon),Turgeon Business Consulting,Issued,2023-11-21 22:30:07+00:00,2024-12-31,Consulting and Management Services,...,BC,CA,NaN,Downtown,1.0,NaN,2026-07-01 02:32:17-07:00,None,NaN,None
3,24,4518848,24-138567,10,Baron Global Financial Canada Ltd,NaN,Issued,2023-12-11 18:12:12+00:00,2024-12-31,Consulting and Management Services,...,BC,CA,V6E 2E9,Downtown,5.0,NaN,2026-07-01 02:32:17-07:00,None,"{'lon': -123.118192793095, 'lat': 49.287734201...",POINT (-123.11819 49.28773)
4,24,4518856,24-138576,10,Jennifer D S Dezell (Jennifer Dezell),Dentons Canada LLP,Issued,2023-12-09 00:34:17+00:00,2024-12-31,Legal Services,...,BC,CA,V6C 3R8,Downtown,217.0,NaN,2026-07-01 02:32:17-07:00,None,"{'lon': -123.113252488858, 'lat': 49.286652613...",POINT (-123.11325 49.28665)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204254,26,4870638,26-148054,00,Shape Property Management Corp,NaN,Cancelled,NaT,NaT,Real Estate Services,...,BC,CA,V7X 1M6,Downtown,35.0,NaN,2026-07-25 00:09:01-07:00,None,"{'lon': -123.118959746657, 'lat': 49.286557071...",POINT (-123.11896 49.28656)
204255,26,4870639,26-148055,00,Bua Group Holdings Ltd,NaN,Issued,2026-01-19 17:29:13+00:00,2026-12-31,Real Estate Services,...,BC,CA,V6C 3A8,Downtown,0.0,324.0,2026-07-25 00:09:01-07:00,None,"{'lon': -123.11758014442, 'lat': 49.2860516234...",POINT (-123.11758 49.28605)
204256,26,4870640,26-148056,00,Shape Holdings Corp,NaN,Cancelled,NaT,NaT,Real Estate Services,...,BC,CA,V7X 1M6,Downtown,0.0,NaN,2026-07-25 00:09:01-07:00,None,"{'lon': -123.118959746657, 'lat': 49.286557071...",POINT (-123.11896 49.28656)
204257,26,4870655,26-148071,00,Coleman Enterprises Corp,NaN,Issued,2025-11-14 19:54:30+00:00,2026-12-31,Real Estate Services,...,BC,CA,V6C 1C8,Downtown,6.0,277.0,2026-07-25 00:09:01-07:00,None,"{'lon': -123.115954794154, 'lat': 49.286140186...",POINT (-123.11595 49.28614)



The shape of the dataframe is (204259, 26)
The columns of the dataframe are ['folderyear', 'licencersn', 'licencenumber', 'licencerevisionnumber', 'businessname', 'businesstradename', 'status', 'issueddate', 'expireddate', 'businesstype', 'businesssubtype', 'unit', 'unittype', 'house', 'street', 'city', 'province', 'country', 'postalcode', 'localarea', 'numberofemployees', 'feepaid', 'extractdate', 'geom', 'geo_point_2d', 'geometry']

Descriptive data


,folderyear,licencersn,licencenumber,licencerevisionnumber,businessname,businesstradename,status,issueddate,expireddate,businesstype,...,province,country,postalcode,localarea,numberofemployees,feepaid,extractdate,geom,geo_point_2d,geometry
count,204259,204259,204259,204259,190777,77378,204259,175491,175515,204259,...,204177,159274,109040,201545,204259.000000,128840.000000,204259,0,102632,102632
unique,3,204255,198655,7,61874,24319,5,NaN,NaN,94,...,48,1,6046,24,NaN,NaN,NaN,0,8974,8974
top,26,5003433,25-102350,00,Parking Corporation of Vancouver,Fasken Martineau DuMoulin LLP,Issued,NaN,NaN,Long-term Rental,...,BC,CA,V5Z 4C2,Downtown,NaN,NaN,NaN,NaN,"{'lon': -123.121463646921, 'lat': 49.285244796...",POINT (-123.121463646921 49.2852447964792)
freq,71391,2,4,147482,503,211,167167,NaN,NaN,45432,...,203097,159274,686,48733,NaN,NaN,NaN,NaN,605,605
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-17 15:09:25.843000+00:00,2025-12-30 18:59:53.908000,NaN,...,NaN,NaN,NaN,NaN,10.021531,516.801428,2026-07-09 11:01:21.312000-07:00,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-11-04 00:55:04+00:00,2024-01-11 00:00:00,NaN,...,NaN,NaN,NaN,NaN,0.000000,2.000000,2026-07-01 02:32:12-07:00,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-01-31 00:56:36.500000+00:00,2024-12-31 00:00:00,NaN,...,NaN,NaN,NaN,NaN,0.000000,207.000000,2026-07-01 02:32:17-07:00,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-12-20 22:40:56+00:00,2025-12-31 00:00:00,NaN,...,NaN,NaN,NaN,NaN,1.000000,277.000000,2026-07-01 02:32:21-07:00,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-11-29 06:15:14.500000+00:00,2026-12-31 00:00:00,NaN,...,NaN,NaN,NaN,NaN,4.000000,405.000000,2026-07-25 00:08:59-07:00,NaN,NaN,NaN
max,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-01 18:30:13+00:00,2027-12-31 00:00:00,NaN,...,NaN,NaN,NaN,NaN,5876.000000,63722.000000,2026-07-25 00:09:02-07:00,NaN,NaN,NaN



Given below: Column, NaN vals, Percentage of dataset
folderyear | 0 | 0.0%.
licencersn | 0 | 0.0%.
licencenumber | 0 | 0.0%.
licencerevisionnumber | 0 | 0.0%.
businessname | 13482 | 6.6004435545067786%.
businesstradename | 126881 | 62.11770350388477%.
status | 0 | 0.0%.
issueddate | 28768 | 14.084079526483533%.
expireddate | 28744 | 14.072329738224509%.
businesstype | 0 | 0.0%.
businesssubtype | 182716 | 89.45309631399351%.
unit | 155310 | 76.0358172712096%.
unittype | 155575 | 76.16555451656964%.
house | 94481 | 46.25548935420226%.
street | 94464 | 46.24716658751879%.
city | 58 | 0.028395321625974863%.
province | 82 | 0.04014510988499895%.
country | 44985 | 22.023509368008266%.
postalcode | 95219 | 46.616795343167254%.
localarea | 2714 | 1.3287052222913067%.
numberofemployees | 0 | 0.0%.
feepaid | 75419 | 36.92322002947239%.
extractdate | 0 | 0.0%.
geom | 204259 | 100.0%.
geo_point_2d | 101627 | 49.75398880832668%.
geometry | 101627 | 49.75398880832668%.


#### Atif's thougts (for now)

- The entire column `geom` is empty. I will drop it because there is nothing useful we can do with it that won't be redundant.
- The `geo_point_2d` and `geometry` is a 1:1 mapping. Same number of missing values, same information, just in different format. I will drop the ones without value, because location clustering without that data will not work.

#### Cleaning up geo data

In [18]:
gdf_geom_intermediate = gdf.drop(columns="geom")
gdf_clean_geo = gdf_geom_intermediate[gdf_geom_intermediate["geometry"].notna()].drop(columns="geometry").reset_index() # need to do reset index, otherwise there are gaps in the dataframe

## Explore how to access the lat-lon from the geometry column
# gdf_clean_geo["geo_point_2d"][0]["lat"]

# Create new columns for lat-lon with just lat and lon. Not "the most efficient" but it makes things easy to calculate - plus we don't have a compute bottleneck, so why not?
gdf_clean_geo["lat"] = gdf_clean_geo["geo_point_2d"].apply(lambda x: x["lat"])
gdf_clean_geo["lon"] = gdf_clean_geo["geo_point_2d"].apply(lambda x: x["lon"])

print(f"Cleaned up GDF with geom removed, and rows with missing geometry removed. {gdf_clean_geo.shape}")
display(gdf_clean_geo)

Cleaned up GDF with geom removed, and rows with missing geometry removed. (102632, 27)


,index,folderyear,licencersn,licencenumber,licencerevisionnumber,businessname,businesstradename,status,issueddate,expireddate,...,province,country,postalcode,localarea,numberofemployees,feepaid,extractdate,geo_point_2d,lat,lon
0,3,24,4518848,24-138567,10,Baron Global Financial Canada Ltd,NaN,Issued,2023-12-11 18:12:12+00:00,2024-12-31,...,BC,CA,V6E 2E9,Downtown,5.0,NaN,2026-07-01 02:32:17-07:00,"{'lon': -123.118192793095, 'lat': 49.287734201...",49.287734,-123.118193
1,4,24,4518856,24-138576,10,Jennifer D S Dezell (Jennifer Dezell),Dentons Canada LLP,Issued,2023-12-09 00:34:17+00:00,2024-12-31,...,BC,CA,V6C 3R8,Downtown,217.0,NaN,2026-07-01 02:32:17-07:00,"{'lon': -123.113252488858, 'lat': 49.286652613...",49.286653,-123.113252
2,5,24,4518857,24-138577,10,Kim Ming Ho Law Corporation,NaN,Issued,2023-11-18 18:41:49+00:00,2024-12-31,...,BC,CA,V7X 1J2,Downtown,1.0,NaN,2026-07-01 02:32:17-07:00,"{'lon': -123.11972958074, 'lat': 49.2861303368...",49.286130,-123.119730
3,6,24,4518858,24-138578,10,Metallic Minerals Corp,NaN,Issued,2023-12-09 20:34:22+00:00,2024-12-31,...,BC,CA,V6C 1T2,Downtown,0.0,NaN,2026-07-01 02:32:17-07:00,"{'lon': -123.114569797322, 'lat': 49.285359392...",49.285359,-123.114570
4,7,24,4518859,24-138579,10,Travlink Employment Consulting and Travel Ltd,NaN,Issued,2023-12-21 21:19:53+00:00,2024-12-31,...,BC,CA,V5R 5J7,Renfrew-Collingwood,3.0,NaN,2026-07-01 02:32:17-07:00,"{'lon': -123.042116815717, 'lat': 49.235652496...",49.235652,-123.042117
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102627,204254,26,4870638,26-148054,00,Shape Property Management Corp,NaN,Cancelled,NaT,NaT,...,BC,CA,V7X 1M6,Downtown,35.0,NaN,2026-07-25 00:09:01-07:00,"{'lon': -123.118959746657, 'lat': 49.286557071...",49.286557,-123.118960
102628,204255,26,4870639,26-148055,00,Bua Group Holdings Ltd,NaN,Issued,2026-01-19 17:29:13+00:00,2026-12-31,...,BC,CA,V6C 3A8,Downtown,0.0,324.0,2026-07-25 00:09:01-07:00,"{'lon': -123.11758014442, 'lat': 49.2860516234...",49.286052,-123.117580
102629,204256,26,4870640,26-148056,00,Shape Holdings Corp,NaN,Cancelled,NaT,NaT,...,BC,CA,V7X 1M6,Downtown,0.0,NaN,2026-07-25 00:09:01-07:00,"{'lon': -123.118959746657, 'lat': 49.286557071...",49.286557,-123.118960
102630,204257,26,4870655,26-148071,00,Coleman Enterprises Corp,NaN,Issued,2025-11-14 19:54:30+00:00,2026-12-31,...,BC,CA,V6C 1C8,Downtown,6.0,277.0,2026-07-25 00:09:01-07:00,"{'lon': -123.115954794154, 'lat': 49.286140186...",49.286140,-123.115955
